# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [10]:
import os
from dotenv import load_dotenv
load_dotenv()
# Get PRICE_DATA path
price_data_path = os.getenv("PRICE_DATA")
print("PRICE_DATA path:", price_data_path)



PRICE_DATA path: ../../05_src/data/prices/


In [11]:
import dask.dataframe as dd

In [16]:
import os
from dotenv import load_dotenv
load_dotenv()
# Get PRICE_DATA path
price_data_path = os.getenv("PRICE_DATA")
print("PRICE_DATA path:", price_data_path)



PRICE_DATA path: ../../05_src/data/prices/


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [17]:
import os
from glob import glob

#  PRICE_DATA directory
PRICE_DATA = os.getenv("PRICE_DATA")

parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)
dd_px = dd.read_parquet(parquet_files).set_index("ticker")
print (parquet_files)



['../../05_src/data/prices\\ACN\\ACN_2001\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2001\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2002\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2002\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2003\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2003\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2004\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2004\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2005\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2005\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2006\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2006\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2007\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2007\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2008\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2008\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2009\\part.0.parque

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [14]:




# Sort data by ticker and Date
dd_px = dd_px.map_partitions(lambda df: df.sort_values(['ticker', 'Date']))

# Using lambda with groupby to create lagged Close
dd_feat = dd_px.map_partitions(
    lambda df: df.assign(
        Close_lag_1 = df.groupby('ticker')['Close'].shift(1),
        Adj_Close_lag_1 = df.groupby('ticker')['Adj Close'].shift(1)
    )
)

# Calculating returns and hi_lo_range in another map_partitions
dd_feat = dd_feat.map_partitions(
    lambda df: df.assign(
        returns = (df['Close'] / df['Close_lag_1']) - 1,
        hi_lo_range = df['High'] - df['Low']
    )
)


print(dd_feat.head())










             Date   Open   High    Low  Close  Adj Close      Volume   source  \
ticker                                                                          
ACN    2001-07-19  15.10  15.29  15.00  15.17  11.404394  34994300.0  ACN.csv   
ACN    2001-07-20  15.05  15.05  14.80  15.01  11.284108   9238500.0  ACN.csv   
ACN    2001-07-23  15.00  15.01  14.55  15.00  11.276587   7501000.0  ACN.csv   
ACN    2001-07-24  14.95  14.97  14.70  14.86  11.171341   3537300.0  ACN.csv   
ACN    2001-07-25  14.70  14.95  14.65  14.95  11.238999   4208100.0  ACN.csv   

        Year  Close_lag_1  Adj_Close_lag_1   returns  hi_lo_range  
ticker                                                             
ACN     2001          NaN              NaN       NaN         0.29  
ACN     2001        15.17        11.404394 -0.010547         0.25  
ACN     2001        15.01        11.284108 -0.000666         0.46  
ACN     2001        15.00        11.276587 -0.009333         0.27  
ACN     2001        14.8

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [15]:

# Convert to pandas dataframe
px_pd = dd_feat.compute()

# Ensure sorting (important for rolling)
px_pd = px_pd.sort_values(["ticker", "Date"])

# Add 10-day moving average of returns per ticker
px_pd["returns_ma_10"] = (
    px_pd.groupby("ticker")["returns"]
    .rolling(10)
    .mean()
    .reset_index(level=0, drop=True)
)

print(px_pd.head(20))



             Date   Open   High    Low  Close  Adj Close      Volume   source  \
ticker                                                                          
ACN    2001-07-19  15.10  15.29  15.00  15.17  11.404394  34994300.0  ACN.csv   
ACN    2001-07-20  15.05  15.05  14.80  15.01  11.284108   9238500.0  ACN.csv   
ACN    2001-07-23  15.00  15.01  14.55  15.00  11.276587   7501000.0  ACN.csv   
ACN    2001-07-24  14.95  14.97  14.70  14.86  11.171341   3537300.0  ACN.csv   
ACN    2001-07-25  14.70  14.95  14.65  14.95  11.238999   4208100.0  ACN.csv   
ACN    2001-07-26  14.95  14.99  14.50  14.50  10.900705   6335300.0  ACN.csv   
ACN    2001-07-27  14.51  14.59  14.50  14.51  10.908223   3524000.0  ACN.csv   
ACN    2001-07-30  14.50  14.78  14.50  14.70  11.051059   3654300.0  ACN.csv   
ACN    2001-07-31  14.71  15.01  14.60  14.96  11.246520   1429000.0  ACN.csv   
ACN    2001-08-01  15.00  15.50  14.90  15.50  11.652478   2087900.0  ACN.csv   
ACN    2001-08-02  15.40  15

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
No; Dask can do rolling operations like pandas.  Converting to pandas was only one option, not a requirement.
+ Would it have been better to do it in Dask? Why?
Yes, if the dataset is big. Dask: good for very large data that doesn’t fit in memory.

(1 pt)

1.No; Dask can do rolling operations like pandas.  Converting to pandas was only one option, not a requirement.
2.Yes, if the dataset is big. Dask: good for very large data that doesn’t fit in memory.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.